# Slide Exercise 05: Context-Aware and Explainable Recommender

This is the refined version of `ContextAware_MovieRecommender.ipynb`.

Learning objectives:
- Build a base content recommendation score.
- Re-rank using context such as device, time, and family mode.
- Explain each recommendation with feature-level reasons.

Main functions used:
- `TfidfVectorizer(...)`: builds item content vectors.
- `cosine_similarity(...)`: computes base relevance.
- `DataFrame.loc[...]`: applies context bonuses to matching rows.
- Custom explanation functions: translate feature overlap into readable reasons.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


Start from a zero-shot style query so the context effect is easy to see.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

movies["context_text"] = movies["genres"].str.replace("|", " ", regex=False) + " " + movies["description"] + " " + movies["keywords"]
vectorizer = TfidfVectorizer(stop_words="english")
item_matrix = vectorizer.fit_transform(movies["context_text"])

def base_recommend(query, n=8):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, item_matrix).ravel()
    results = movies[["title", "genres", "director", "duration_min", "family_friendly"]].copy()
    results["base_score"] = scores
    return results.sort_values("base_score", ascending=False).head(n)

base_recommend("family adventure comedy")


Apply context rules as a transparent re-ranking layer.


In [ ]:
def apply_context(results, context):
    reranked = results.copy()
    reranked["context_bonus"] = 0.0

    if context == "morning_mobile":
        reranked.loc[reranked["duration_min"] <= 110, "context_bonus"] += 0.12
    if context == "evening_tv":
        reranked.loc[reranked["duration_min"] >= 120, "context_bonus"] += 0.10
    if context == "family_mode":
        reranked.loc[reranked["family_friendly"] == 1, "context_bonus"] += 0.25

    reranked["final_score"] = reranked["base_score"] + reranked["context_bonus"]
    return reranked.sort_values("final_score", ascending=False)

base = base_recommend("family adventure comedy")
apply_context(base, "family_mode")


Generate short explanations from shared query terms and metadata.


In [ ]:
def explain(query, title):
    movie = movies[movies["title"].eq(title)].iloc[0]
    q = set(query.lower().split())
    genre_matches = [g for g in movie["genres"].split("|") if g.lower() in q]
    keyword_matches = [w for w in movie["keywords"].split() if w.lower() in q]
    reasons = genre_matches + keyword_matches
    if movie["family_friendly"] == 1:
        reasons.append("family-friendly")
    return "Recommended because it shares: " + ", ".join(reasons or ["related content"])

top = apply_context(base, "family_mode").iloc[0]["title"]
explain("family adventure comedy", top)


Interpretation:

Context-aware recommendation does not replace the base recommender. It adjusts a reasonable ranking for the user's current situation.

Student task:
1. Add a `short_break` context that favors movies under 100 minutes.
2. Explain one recommendation before and after re-ranking.
